In [23]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('comm_log.db')


In [31]:
query = "select * from communication_log limit 20;"
df = pd.read_sql_query(query, conn)
df

,id,merchant_id,communication_id,customer_id,communication_type,delivery_status,sent_time,scheduled_time,credit_used,channel
0,1,501,9001,C1,2,900,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
1,2,501,9001,C2,2,1100,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
2,3,501,9002,C2,2,900,2026-10-04 10:00:00,2026-10-04 10:00:00,1,sms
3,4,501,9001,C3,2,1100,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
4,5,501,9002,C3,2,1100,2026-10-04 10:00:00,2026-10-04 10:00:00,1,sms
5,6,501,9003,C3,2,900,2026-10-05 10:00:00,2026-10-05 10:00:00,1,sms
6,7,501,9001,C4,2,900,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
7,8,501,9001,C5,2,900,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
8,9,501,9001,C6,2,900,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
9,10,501,9001,C7,2,900,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms


In [30]:
query = "select * from campaign limit 10;"
df = pd.read_sql_query(query, conn)
df

,id,merchant_id,parent_id,name,creation_status,processing_status
0,9001,501,NaN,Diwali Cart Recovery - Wave 1,approved,processed
1,9002,501,9001.0,Diwali Cart Recovery - Retry A,approved,processed
2,9003,501,9002.0,Diwali Cart Recovery - Retry B,approved,processed
3,9004,501,9001.0,Diwali Cart Recovery - Retry C (pending),approval_awaiting,processed
4,9101,501,NaN,Diwali Flash Sale - Standalone,approved,processed
5,9201,501,NaN,Diwali Wave 2,approved,processed
6,9202,501,9201.0,Diwali Wave 2 - Retry,approved,processed


In [ ]:
query="SELECT COUNT(*) AS total_rows FROM communication_log;"        #
df = pd.read_sql_query(query, conn)
df


,total_rows
0,30


In [48]:
# Number of qualifying campaigns
query="""SELECT COUNT(*) AS count
FROM campaign
WHERE creation_status IN ('approved', 'aborted', 'resumed', 'stopped')
  AND processing_status = 'processed';"""
df=pd.read_sql_query(query,conn)
df

,count
0,6


In [47]:
# Count communication logs for those qualifying campaigns
query="""SELECT COUNT(*) AS count
FROM communication_log
WHERE communication_id IN (
      SELECT id
      FROM campaign
      WHERE creation_status IN ('approved', 'aborted', 'resumed', 'stopped')
        AND processing_status = 'processed'
  );"""
df=pd.read_sql_query(query,conn)
df

,count
0,26


In [ ]:
# Check delivery status
query="""SELECT delivery_status, COUNT(*) AS count
FROM communication_log
WHERE communication_id IN (
    SELECT id
    FROM campaign
    WHERE creation_status IN ('approved', 'aborted', 'resumed', 'stopped')
      AND processing_status = 'processed'
)
GROUP BY delivery_status;"""
df=pd.read_sql_query(query,conn)
df

,delivery_status,count
0,900,22
1,1100,4


In [ ]:
# Final Query 
query="""SELECT COUNT(*) AS target_base
FROM communication_log cl
JOIN campaign c
  ON cl.communication_id = c.id
WHERE c.creation_status IN ('approved', 'aborted', 'resumed', 'stopped')
  AND c.processing_status = 'processed'
  AND c.parent_id IS NULL;"""
df=pd.read_sql_query(query,conn)
df

,target_base
0,22


In [ ]:
# Identify eligible campaigns and their retry roots
query = """WITH RECURSIVE campaign_chain AS (
    SELECT
        id AS campaign_id,
        id AS root_id,
        parent_id
    FROM campaign
    WHERE merchant_id = 501

    UNION ALL

    SELECT
        cc.campaign_id,
        c.id AS root_id,
        c.parent_id
    FROM campaign_chain cc
    JOIN campaign c
        ON cc.parent_id = c.id
)

SELECT
    campaign_id,
    root_id
FROM campaign_chain
WHERE parent_id IS NULL;"""

df = pd.read_sql_query(query, conn)
df


,campaign_id,root_id
0,9001,9001
1,9101,9101
2,9201,9201
3,9002,9001
4,9004,9001
5,9202,9201
6,9003,9001


In [ ]:
# Attach sends to the underlying communication
query = """WITH RECURSIVE campaign_chain AS (
    SELECT
        id AS campaign_id,
        id AS root_id,
        parent_id
    FROM campaign
    WHERE merchant_id = 501

    UNION ALL

    SELECT
        cc.campaign_id,
        c.id AS root_id,
        c.parent_id
    FROM campaign_chain cc
    JOIN campaign c
        ON cc.parent_id = c.id
),

resolved_campaigns AS (
    SELECT
        campaign_id,
        root_id
    FROM campaign_chain
    WHERE parent_id IS NULL
)

SELECT
    rc.root_id,
    cl.communication_id,
    cl.customer_id,
    cl.delivery_status
FROM communication_log cl
JOIN resolved_campaigns rc
    ON cl.communication_id = rc.campaign_id
JOIN campaign c
    ON cl.communication_id = c.id
WHERE cl.merchant_id = 501
  AND cl.communication_type = '2'
  AND cl.sent_time >= '2026-10-01'
  AND cl.sent_time < '2026-11-01'
  AND c.creation_status IN (
      'approved',
      'aborted',
      'resumed',
      'stopped'
  )
  AND c.processing_status = 'processed'
ORDER BY rc.root_id, cl.customer_id;"""


df = pd.read_sql_query(query, conn)
df

,root_id,communication_id,customer_id,delivery_status
0,9001,9001,C1,900
1,9001,9001,C10,900
2,9001,9001,C2,1100
3,9001,9002,C2,900
4,9001,9001,C3,1100
5,9001,9002,C3,1100
6,9001,9003,C3,900
7,9001,9001,C4,900
8,9001,9001,C5,900
9,9001,9001,C6,900


In [ ]:
# Calculate target_base
query = """WITH RECURSIVE campaign_chain AS (
    SELECT
        id AS campaign_id,
        id AS root_id,
        parent_id
    FROM campaign
    WHERE merchant_id = 501

    UNION ALL

    SELECT
        cc.campaign_id,
        c.id AS root_id,
        c.parent_id
    FROM campaign_chain cc
    JOIN campaign c
        ON cc.parent_id = c.id
),

resolved_campaigns AS (
    SELECT
        campaign_id,
        root_id
    FROM campaign_chain
    WHERE parent_id IS NULL
),

eligible_sends AS (
    SELECT
        cl.id AS send_id,
        cl.customer_id,
        cl.communication_id,
        rc.root_id
    FROM communication_log cl
    JOIN resolved_campaigns rc
        ON cl.communication_id = rc.campaign_id
    JOIN campaign c
        ON cl.communication_id = c.id
    WHERE cl.merchant_id = 501
      AND cl.communication_type = '2'
      AND cl.sent_time >= '2026-10-01'
      AND cl.sent_time < '2026-11-01'
      AND c.creation_status IN (
          'approved',
          'aborted',
          'resumed',
          'stopped'
      )
      AND c.processing_status = 'processed'
),

root_summary AS (
    SELECT
        root_id,
        COUNT(DISTINCT communication_id) AS campaign_count
    FROM eligible_sends
    GROUP BY root_id
)

SELECT
    SUM(
        CASE
            WHEN rs.campaign_count = 1
                THEN (
                    SELECT COUNT(*)
                    FROM eligible_sends e2
                    WHERE e2.root_id = rs.root_id
                )
            ELSE (
                SELECT COUNT(DISTINCT e2.customer_id)
                FROM eligible_sends e2
                WHERE e2.root_id = rs.root_id
            )
        END
    ) AS target_base
FROM root_summary rs;"""

df = pd.read_sql_query(query, conn)
df


,target_base
0,22
